In [ ]:
#Carga de los archivos a procesar, pueden ser uno o varios

ARCHIVO = "/DECM_Corpus/DECM_Machine_Ready_Corpus/1_RG_Guatemala _Acuna.txt"

In [ ]:
from ollama import chat 
import os
import requests
import json
from collections import defaultdict

In [ ]:
#Procesando y leyendo los textos integrados

with open(ARCHIVO, "r", encoding="utf-8") as f:
    texto = f.read()

print("len(texto) =", len(texto))
print(texto[:300])


In [ ]:
#División de textos en chunks para un procesamiento más eficiente

def chunk_text(text, max_chars=1500):
    """
    Versión simple: corta el texto cada max_chars caracteres,
    sin buscar puntos ni nada. Es muy rápida y no se traba.
    """
    return [text[i:i+max_chars] for i in range(0, len(text), max_chars)]


In [ ]:
#Analizando los resultados y cantidad de chunks por texto 

%time chunks = chunk_text(texto)
print("len(texto) =", len(texto))
print("n chunks =", len(chunks))
print("Ejemplo chunk 0:\n", chunks[0][:200])


In [ ]:
# Modelo ollama
MODEL_ONTO="llama3:latest"
OLLAMA_HOST = "http://localhost:11434"



def chat_ollama(system_prompt, user_prompt, model=MODEL_ONTO):
    url = f"{OLLAMA_HOST}/api/chat"
    resp = requests.post(url, json={
        "model": model,
        "stream": False,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    })
    resp.raise_for_status()
    data = resp.json()
    return data["message"]["content"]


In [ ]:
#Desarrollo del prompt y categorías
#Aquí se detalla la estructura de los resultados que queremos, así como las entradas que debemos de tener
#Que es y que debe hacer el modelo

def extraer_ontologia_de_chunk(texto_chunk):

    # === SPECIFICATION SECTION (inspirado en Spec Kit) ===
    SPEC = {
        "entity_schema": {
            "id": "string (c1, c2, ...)",
            "label": "string",
            "type": "one of predefined categories",
            "aliases": "list of strings"
        },
        "relation_schema": {
            "subject": "string (entity id)",
            "predicate": "one of predefined predicates",
            "object": "string (entity id)",
            "evidence": "string (short quote)"
        },
        "valid_categories": [
            "persona", "lugar", "pueblo_indigena", "toponimo",
            "institucion_colonial", "cargo", "autoridad", "documento",
            "obra", "evento_historico", "grupo_social", "costumbre",
            "practica_cultural", "actividad_economica", "territorio",
            "familia", "linaje", "comunidad", "autor", "testigo"
        ],
        "valid_predicates": [
            "menciona_a", "describe_a", "autor_de", "participa_en",
            "ocurre_en", "ubicado_en", "pertenece_a",
            "es_parte_de", "relacionado_con"
        ]
    }


    # === SYSTEM PROMPT USING SPEC ===
    system_prompt = f"""
Eres una asistente experta en análisis semántico de textos coloniales.
Debes seguir una ESPECIFICACIÓN ESTRICTA (Spec-Driven Development).

=== SPECIFICACIÓN ===
- Solo puedes usar las categorías: {", ".join(SPEC["valid_categories"])}
- Solo puedes usar los predicados: {", ".join(SPEC["valid_predicates"])}
- Si un concepto no encaja perfectamente, elige la categoría más cercana.
- No inventes categorías ni predicados.
- Sigue exactamente estos esquemas JSON:

Entidad:
{SPEC["entity_schema"]}

Relación:
{SPEC["relation_schema"]}

Reglas:
1. El JSON generado debe ser **válido**.
2. No escribas texto fuera del JSON.
3. Usa solo contenido del fragmento.
4. Cada entidad debe tener un ID (c1, c2...).
5. La evidencia en relaciones debe ser una cita del fragmento.
6. No dupliques entidades en un mismo chunk.
7. No interpretes más allá del texto.

Tu rol: ejecutar la especificación, no 'ser creativo'.
"""


    # === USER PROMPT ===
    user_prompt = f"""
Analiza el siguiente fragmento usando la especificación:

\"\"\"{texto_chunk}\"\"\"


=== TAREA ===
1. Extrae todas las entidades relevantes según las categorías válidas.
2. Extrae todas las relaciones permitidas usando los predicados válidos.
3. Sigue EXACTAMENTE la especificación.
4. Devuelve SOLO el JSON con forma:

{{
  "concepts": [ ... ],
  "relations": [ ... ]
}}

No incluyas comentarios, explicaciones ni texto adicional.
    """


    # === EXECUTE MODEL ===
    raw = chat_ollama(system_prompt, user_prompt).strip()


    # === JSON PARSING SAFETY ===
    try:
        data = json.loads(raw)
    except:
        start = raw.find("{")
        end = raw.rfind("}")
        if start != -1 and end != -1:
            try:
                data = json.loads(raw[start:end+1])
            except:
                print("⚠️ JSON inválido en este chunk.")
                data = {"concepts": [], "relations": []}
        else:
            print("⚠️ Respuesta sin JSON claro.")
            data = {"concepts": [], "relations": []}

    # === MINIMUM GUARANTEES ===
    data.setdefault("concepts", [])
    data.setdefault("relations", [])

    return data


In [ ]:
# Ontología global, si existen palabras reptidas, contexto, etc se acumula aquí
concept_id_counter = 1
concept_index = {}         
concepts_global = {}       
relations_global = []      

def normalizar_label(label):
    return " ".join(label.lower().strip().split())

for i, ch in enumerate(chunks):
    print(f"Procesando chunk {i+1}/{len(chunks)}...")
    ont_chunk = extraer_ontologia_de_chunk(ch)

    #  Integrando los conceptos
    local_to_global = {}  # mapeo ids locales (c1, c2...) a ids globales

    for c in ont_chunk["concepts"]:
        label = c.get("label", "").strip()
        if not label:
            continue
        key = normalizar_label(label)
        if key in concept_index:
            global_id = concept_index[key]
        else:
            global_id = f"C{concept_id_counter}"
            concept_id_counter += 1
            concept_index[key] = global_id
            # guardar concepto global
            concepts_global[global_id] = {
                "id": global_id,
                "label": label,
                "type": c.get("type", "desconocido"),
                "aliases": c.get("aliases", [])
            }
        local_to_global[c.get("id", "")] = global_id

    # Integrando las relaciones usando el mapeo local_to_global
    for r in ont_chunk["relations"]:
        subj_local = r.get("subject")
        obj_local  = r.get("object")

        # 🔧 Si vienen como listas, tomar el primer elemento
        if isinstance(subj_local, list):
            subj_local = subj_local[0] if subj_local else None
        if isinstance(obj_local, list):
            obj_local = obj_local[0] if obj_local else None

        # 🔧 Si no son strings o vienen vacíos, saltar la relación
        if not isinstance(subj_local, str) or not isinstance(obj_local, str):
            continue

        if subj_local not in local_to_global or obj_local not in local_to_global:
            continue

        subj_global = local_to_global[subj_local]
        obj_global  = local_to_global[obj_local]

        relations_global.append({
            "subject": subj_global,
            "predicate": r.get("predicate", "relacion_desconocida"),
            "object": obj_global,
            "evidence": r.get("evidence", ""),
            "source_chunk": i  # por si luego quieres regresar al texto
        })

print("\n✅ Ontología construida.")
print(f"- Conceptos totales: {len(concepts_global)}")
print(f"- Relaciones totales: {len(relations_global)}")



In [ ]:
#Relaciones encontradas entre las palabras 

for cid, c in list(concepts_global.items())[:20]:  # muestra solo 20
    print(f"{cid}: {c['label']} (tipo: {c['type']})  aliases={c.get('aliases', [])}")

In [ ]:
#Guardando los resultados en un txt para poder consultarlos y analizarlos

ruta = "/Downloads/ontologia_por_categorias_final.txt"

with open(ruta, "w", encoding="utf-8") as f:
    for tipo, items in sorted(por_tipo.items()):
        f.write(f"=== {tipo} ({len(items)}) ===\n")

        for item in items:
            cid = item.get("id", "???")
            label = item.get("label", "")
            f.write(f"  - {cid}: {label}\n")

        f.write("\n")



In [ ]:

# Agrupando las categorías encontradas 

por_tipo = defaultdict(list)
for cid, c in concepts_global.items():
    tipo = c.get("type", "desconocido")
    por_tipo[tipo].append(c)

# Imprimir resumen
print("=== RESUMEN DE LA ONTOLOGÍA ===\n")
print(f"Total de categorías: {len(por_tipo)}\n")
total_conceptos = 0

for tipo, items in sorted(por_tipo.items()):
    print(f"- {tipo}: {len(items)} conceptos")
    total_conceptos += len(items)

print("\nTotal de conceptos en toda la ontología:", total_conceptos)



In [ ]:
#Guardando las categorías agrupadas para consulta

lineas = []
lineas.append("=== RESUMEN DE LA ONTOLOGÍA ===\n")
lineas.append(f"Total de categorías: {len(por_tipo)}\n")

total_conceptos = 0
for tipo, items in sorted(por_tipo.items()):
    lineas.append(f"{tipo}: {len(items)} conceptos")
    total_conceptos += len(items)

lineas.append(f"\nTotal de conceptos en toda la ontología: {total_conceptos}")

ruta = "/Downloads/resumen_ontologia.txt"

with open(ruta, "w", encoding="utf-8") as f:
    f.write("\n".join(lineas))

